# **The 6-step Data Quality Method**
## https://github.com/royruddle/6-step-data-quality-method

This is the notebook that is used in the YouTube video about **Step 1** of the method (https://www.youtube.com/watch?v=m7zjU5ojoBo), and shows how to use the vizdataquality package to investigate data quality.

The 6 steps are:
1. Is anything obviously wrong?
2. Watch out for special values
3. Is any data missing?
4. Check each variable
5. Check combinations of variables
6. Profile the cleaned data

## **Step 1: Is anything obviously wrong (look at your data and any documentation)?**

This step uses off-street parking fines data ('Quarter 4 201819.csv'; https://datamillnorth.org/dataset/v8ggw/off-street-parking-fines), (c) Leeds City Council, 2019. The data is licensed under the terms of the Open Government Licence (https://www.nationalarchives.gov.uk/doc/open-government-licence/version/2/).

## Includes: vizdataquality and other libraries

In [1]:
import os
import pandas as pd

from vizdataquality import calculate as vdqc, plot as vdqp, utils as vdqu

## File format

In [2]:
input_filename = os.path.join('../../examples', 'Quarter 4 201819.csv')
encoding_results = vdqu.detect_file_encoding(input_filename, read_in_chunks=False)
print(encoding_results)

{'encoding': 'ascii', 'confidence': 1.0, 'language': 'ms', 'mime_type': 'text/plain'}


## Read the datafile

In [3]:
# Make sure only blank cells are flagged as missing values and read the whole datafile is read to determine datatypes.
kwargs = {'na_values': '', 'keep_default_na': False, 'low_memory': False}

# If you know the column separator then you can specify it in the read_csv() call. Otherwise Pandas will try to infer it.
df = pd.read_csv(input_filename, encoding=encoding_results['encoding'], **kwargs)

### Datafile statistics

Use vizdataquality convenience function

In [4]:
datafile_stats = vdqc.step1_datafile_stats(encoding_results, input_filename, df)
print(datafile_stats)

                Statistic                   Value
0                Encoding  ascii (confidence=1.0)
1            Size (bytes)                  753976
2          Number of rows                    7108
3       Number of columns                       9
4  Number of blank values                    7108


### Example values

In [5]:
# First and last rows
print(df.iloc[0], '\n')
print(df.iloc[-1])

PCN                                        LS33109561
ISSUED                                     03/01/2009
LOCATION                        THE MARKETS CP - CITY
CONTRAVENTION    83 WITHOUT DISPLAYING A VALID TICKET
FINE                                               50
Last Pay Date                              28/03/2019
Total Paid                                        0.0
Balance                                           0.0
Unnamed: 8                                        NaN
Name: 0, dtype: object 

PCN                                        LS08541469
ISSUED                                     31/03/2019
LOCATION                        WEST STREET CP - CITY
CONTRAVENTION    83 WITHOUT DISPLAYING A VALID TICKET
FINE                                               50
Last Pay Date                              31/03/2019
Total Paid                                        0.0
Balance                                           0.0
Unnamed: 8                                        NaN
Nam

### Investigate that strange column

In [6]:
column = 'Unnamed: 8'
print(column, 'has', df[column].isna().sum(), 'missing values and', df[column].notnull().sum(), 'present values')

Unnamed: 8 has 7108 missing values and 0 present values


### Alternatively, use vizdataquality convenience function

In [7]:
issues_step1 = vdqc.step1_issues(df)

for row in issues_step1.itertuples(index=False):
    print(row[0].rjust(30) + ': ', row[1])

Number of missing column names:  1
       Number of empty columns:  1
                  Extra column:  Occurs because every row ends with the separator (e.g., ',')


### Clean the data
- Omit the strange column
- Specify the correct format for the date columns

In [8]:
columns = df.columns[:-1]
df = pd.read_csv(input_filename, usecols=columns, parse_dates=['ISSUED', 'Last Pay Date'], dayfirst=True, encoding=encoding_results['encoding'])